# Place a signed Perps order

Required environment variables: `EXCHANGEOS_API_KEY`, `EXCHANGEOS_API_SECRET`, `EXCHANGEOS_API_PASSPHRASE`, `EXCHANGEOS_AGENT_PRIVATE_KEY`, `EXCHANGEOS_ASSET_ID`.

Optional: `EXCHANGEOS_ORDER_PRICE`, `EXCHANGEOS_ORDER_SIZE`.

**This notebook submits a real order. Review all values before running the final cell.**

In [ ]:
import os

from exchangeos_sdk import (
    Credentials,
    ExchangeOSClient,
    MarginMode,
    OrderSide,
    OrderTimeInForce,
    PlaceOrder,
    limit_order_type,
    order_item,
    perp_market,
)

credentials = Credentials(
    api_key=os.environ["EXCHANGEOS_API_KEY"],
    secret_key=os.environ["EXCHANGEOS_API_SECRET"],
    passphrase=os.environ["EXCHANGEOS_API_PASSPHRASE"],
)

In [ ]:
# order_item() generates a valid API clientOrderId when it is omitted.
order = order_item(
    asset_id=os.environ["EXCHANGEOS_ASSET_ID"],
    side=OrderSide.BUY,
    market_type=perp_market(MarginMode.CROSS),
    price=os.getenv("EXCHANGEOS_ORDER_PRICE", "50000"),
    size=os.getenv("EXCHANGEOS_ORDER_SIZE", "0.001"),
    order_type=limit_order_type(OrderTimeInForce.GTC),
)
action = PlaceOrder(orders=[order])

In [ ]:
with ExchangeOSClient(auth=credentials) as client:
    signed_transaction = client.sign_action(
        action,
        os.environ["EXCHANGEOS_AGENT_PRIVATE_KEY"],
    )
    result = client.trade.place_order(signed_transaction)

print({"clientOrderId": order["clientOrderId"], "response": result})